## Setting environment up for Colab

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
base_path = Path('/content/drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/week_1/sessions/')

# Week 1 – Session 3 - Part b: Data Cleaning & Basic Data Transformations

In this part of Week 1, we will build a **practical, reliable data cleaning pipeline** for the FakeNewsNet dataset. This step is essential to ensure that any machine learning model we build later is learning from **clean, consistent, and meaningful data**.

---

## Why This Matters

Dirty, incomplete, or inconsistent data can introduce:
- Biases (e.g., models overfitting to artifacts like article length or dataset source-specific quirks)
- Poor generalization (e.g., patterns that don’t hold up in real-world news)
- Broken features (e.g., missing dates, duplicate articles)

A robust cleaning pipeline also helps for more advanced techniques, e.g. when we integrate our data into **retrieval-augmented generation (RAG)** systems, where clean and well-structured metadata (like `dataset_source` and `date`) is critical for retrieving the right supporting facts.

---

## Objectives

1. Remove:
    - **Handle Missing Field Values:**  
        - Detect and drop rows with critical missing fields (`article` content).
    - **Outlier & Duplicate Detection:**  
        - Identify extremely short or unusually long articles that may not represent typical news.
        - Identify and remove duplicates that may introduce bias.
        - Decide whether to remove or truncate these outliers.
1. Impute:
    - **Handle Missing Field Values:**  
        - Impute or flag non-critical fields (e.g., `date` or `author`) when needed.
1. Transform:
    - **Source Domain Extraction**
    - **Basic Categorical Encoding:**  
        - Encode `dataset_source` columns for modeling.
    - **Date & URL Parsing**
    - **Text Cleaning:**  
        - Remove empty or whitespace-only articles.
        - Optionally strip unwanted characters or fix encoding issues.

---

> **Tip:** Keeping cleaning steps modular and reproducible is essential — they form the backbone of any production-ready system.

## Load FakeNewsNet Data

We already added the `load_raw_fakenewsnet_data` function in the previous part, and we can just immediately reuse it if we make sure to install our package as an editable package

In [3]:
# TODO: Confirm this works in colab
!pip install -e {base_path}/../../src/

Obtaining file:///content/drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/src
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for misinformation_detection (pyproject.toml) ... done
  Created wheel for misinformation_detection: filename=misinformation_detection-1.0.0-0.editable-py3-none-any.whl size=3582 sha256=c2a3b427ffcf998b8a1912f8986bac24da61c68926241d487ea5a4fd646bafa8
  Stored in directory: /tmp/pip-ephem-wheel-cache-5863xivd/wheels/46/67/19/6967c24d7ffe355b16d9bd45b2a6576a3fa548c7b58c981aee
Successfully built misinformation_detection


In [4]:
from misinformation_detection.data import load_fakenewsnet_data
from pathlib import Path

DATA_DIR = base_path / Path("../../data/raw/FakeNewsNet")

fnn_df = load_fakenewsnet_data(DATA_DIR, verbose=True)
fnn_df.sample(3)

Loaded 23,672 articles from GossipCop and PolitiFact


,id,title,article,url,date,dataset_source,label
2134,gossipcop-3098346462,Ellen DeGeneres and Portia de Rossi Exchange L...,Happy anniversary Ellen DeGeneres and Portia d...,people.com/tv/ellen-degeneres-portia-de-rossi-...,NaN,gossipcop,1
7741,gossipcop-942155,'Gladiator' Cast Reunites at Rome's Colosseum ...,Rome hosted a very special screening for the 1...,https://www.hollywoodreporter.com/news/gladiat...,2018-06-06,gossipcop,real
1299,gossipcop-1369753636,AWKWARD: Chelsea Clinton tries ‘bonding’ with ...,Not sure Megyn should worry TOO much about the...,twitchy.com/samj-3930/2017/06/11/awkward-chels...,2017-06-11,gossipcop,fake


We will also fix the inconsistency in our labels identified last notebook

In [5]:
fnn_df['label'] = fnn_df['label'].replace({'0': 'real', '1': 'fake'})

## Step 1: Remove

As we said earlier, the first step in the protocol is removing duplicates, irrelevant features, corrupt data, etc.

### a. Remove Rows with Empty Articles

The `article` field is the core text our models will analyze. Any rows with missing or empty `article` content must be removed to ensure we only train on valid, meaningful data.

We’ll:
1. Count rows with `NaN` or empty/whitespace-only `article` fields.
2. Drop these rows.
3. Verify that no empty articles remain.

In [6]:
# Count missing or empty articles
num_missing = fnn_df["article"].isnull().sum()
num_empty = (fnn_df["article"].astype(str).str.strip() == "").sum()

print(f"Missing articles: {num_missing}")
print(f"Empty articles: {num_empty}")

# Remove rows with missing or empty articles (in-place)
fnn_df = fnn_df[
    fnn_df["article"].notnull() &
    (fnn_df["article"].astype(str).str.strip() != "")
].copy()

print(f"Remaining rows after removal: {len(fnn_df):,}")

# Double-check
print("Check for any remaining missing or empty articles:")
print(fnn_df["article"].isnull().sum())
print((fnn_df["article"].astype(str).str.strip() == "").sum())

Missing articles: 7244
Empty articles: 2
Remaining rows after removal: 16,426
Check for any remaining missing or empty articles:
0
0


### b. Remove Article Length Outliers

Article length can vary naturally, but extreme outliers can hurt our model by introducing noise or trivial correlations. We looked into article lenghts last session and identified the following:

- **Very short articles** (<250 characters) often contain little usable information.
- **Extremely long articles** (>30,000 characters) may be scraped dumps or merged documents.

We’ll filter these cases to keep only reasonably sized, high-quality articles.

In [7]:
fnn_df["article_length"] = fnn_df["article"].astype(str).apply(len)

# Visual check
print(fnn_df["article_length"].describe())

# Filter out short and long outliers
fnn_df = fnn_df[
    (fnn_df["article_length"] >= 250) &
    (fnn_df["article_length"] <= 30000)
].copy()

print(f"Remaining rows after length filtering: {len(fnn_df):,}")

fnn_df.drop(columns=["article_length"], inplace=True)

count     16426.000000
mean       4052.284792
std        8468.975594
min          21.000000
25%        1216.000000
50%        2195.000000
75%        3617.750000
max      337667.000000
Name: article_length, dtype: float64
Remaining rows after length filtering: 14,944


### c. Check & Remove Duplicate Articles

Before training our model, it’s important to **identify and remove duplicate entries**  
based on the combination of **`title` and `article`**.

---

#### Why This Matters

- Duplicate articles can **bias our model**, especially if the same text appears in both training and test sets.
- Removing them ensures our evaluation metrics are **more honest and robust**.

---

#### What Happens Here

- Count how many duplicates exist (`keep=False` shows all copies).
- Drop all but the **first occurrence**.
- Verify that no duplicates remain.

In [8]:
# Print number of duplicate entries
duplicate_entries = fnn_df[fnn_df.duplicated(subset=["title", "article"], keep=False)]
print(f"Duplicate entries: {len(duplicate_entries)}")

fnn_df = fnn_df.drop_duplicates(subset=["title", "article"], keep="first")
duplicate_entries = fnn_df[fnn_df.duplicated(subset=["title", "article"], keep=False)]
print(f"Duplicate entries after: {len(duplicate_entries)}")

Duplicate entries: 1856
Duplicate entries after: 0


## Step 2: Impute

In the second step, imputation, we need to now fill in missing data **strategically** (i.e. we should make sure to preserve data relationships without introducing a source of noise or bias)

### a. Handle Missing URLs

The `url` field is not strictly necessary for training text models but is helpful for traceability and auditing.  
Instead of dropping rows with missing URLs, we’ll fill them with a default placeholder.

This ensures downstream pipelines don’t fail due to `NaN` values.

In [9]:
# Check how many missing URLs
missing_urls = fnn_df["url"].isnull().sum()
print(f"Missing URLs: {missing_urls}")

# Fill with default placeholder
fnn_df["url"] = fnn_df["url"].fillna("UNKNOWN_URL")

# Double-check
print("Check for remaining missing URLs:")
print(fnn_df["url"].isnull().sum())

Missing URLs: 0
Check for remaining missing URLs:
0


### b. Handle Missing Dates

The `date` field can be useful for analyzing misinformation trends over time or for retrieval tasks.

To keep the dataset consistent, we’ll fill missing dates with a clear placeholder (`"UNKNOWN_DATE"`).  
This prevents downstream errors and makes it obvious when the real date is not available.

In [10]:
# Check missing dates
missing_dates = fnn_df["date"].isnull().sum()
print(f"Missing dates: {missing_dates}")

# Fill with placeholder
fnn_df["date"] = fnn_df["date"].fillna("UNKNOWN_DATE")

# Double-check
print("Check for remaining missing dates:")
print(fnn_df["date"].isnull().sum())

Missing dates: 4009
Check for remaining missing dates:
0


## Step 3: Transform

The final step in our protocol is transformation, and it involves Text pre-processing, URL extraction, date parsing, etc.

### a. Extract Source Domains from URLs

Having a clean `source_domain` feature helps analyze where articles come from and can be useful in downstream tasks like source credibility scoring or retrieval.

We’ll:
- Extract the domain name from each `url`
- Treat `"UNKNOWN_URL"` and empty strings as `"unknown"`
- Handle invalid or malformed URLs gracefully

In [11]:
from urllib.parse import urlparse

def extract_domain(url):
    # Treat placeholder as unknown
    if url == "UNKNOWN_URL" or not isinstance(url, str) or url.strip() == "":
        return "unknown"
    try:
        cleaned = url.strip()
        if not cleaned.startswith(("http://", "https://")):
            cleaned = "https://" + cleaned  # default scheme
        netloc = urlparse(cleaned).netloc
        return netloc.replace("www.", "") if netloc else "invalid"
    except:
        return "invalid"

# Apply extraction
fnn_df["source_domain"] = fnn_df["url"].apply(extract_domain)

# Preview top domains
print("Top source domains:")
print(fnn_df["source_domain"].value_counts().head(10))

Top source domains:
source_domain
people.com               1314
dailymail.co.uk           897
usmagazine.com            643
etonline.com              617
en.wikipedia.org          436
hollywoodreporter.com     299
variety.com               271
ew.com                    195
radaronline.com           182
elle.com                  177
Name: count, dtype: int64


### b. Source Domain One-Hot Encoding
---

#### Why One-Hot Encoding?

- `source_domain` is a **categorical feature** — there’s **no inherent ordinal relationship** between domains like `people.com`, `variety.com`, or `gossipcop`.
- If we assigned them numerical labels (`0, 1, 2, ...`), the model might wrongly assume there’s a meaningful ranking — but there isn’t!
- **One-hot encoding** preserves the idea that each domain is just a distinct category, not ordered or continuous.

---
News articles come from many different domains — some sources appear often, others only a few times.  
Rather than treating every single domain separately (which would create lots of sparse columns),  
we’ll use **one-hot encoding** for only the **top 15 most frequent domains**.  
All other rare domains will be grouped together as **“Other”**.

In [12]:
import pandas as pd

# Count top 15 domains
top_domains = fnn_df["source_domain"].value_counts().nlargest(15).index.tolist()

# Assign 'Other' to all less common domains
fnn_df["source_domain_grouped"] = fnn_df["source_domain"].apply(
    lambda x: x if x in top_domains else "Other"
)

# One-hot encode the grouped domain column
source_dummies = pd.get_dummies(fnn_df["source_domain_grouped"], prefix="source")

# Add to main DataFrame
fnn_df = pd.concat([fnn_df, source_dummies], axis=1)

# Preview the new columns
fnn_df.filter(like="source_").head(3)

,source_domain,source_domain_grouped,source_Other,source_billboard.com,source_dailymail.co.uk,source_elle.com,source_en.wikipedia.org,source_etonline.com,source_ew.com,source_harpersbazaar.com,source_hollywoodreporter.com,source_inquisitr.com,source_people.com,source_radaronline.com,source_thewrap.com,source_today.com,source_usmagazine.com,source_variety.com
0,dailymail.co.uk,dailymail.co.uk,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
2,variety.com,variety.com,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3,dailymail.co.uk,dailymail.co.uk,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False


### c. Clean the Article Text

Raw text often contains extra noise:
- Unwanted HTML tags
- URLs embedded in the text
- Non-ASCII or control characters
- Irregular whitespace

These patterns can hurt downstream tokenization and feature extraction.  
We’ll apply a simple cleaning function that:
1. Lowercases the text
2. Strips leading/trailing spaces
3. Removes embedded URLs and HTML
4. Replaces non-ASCII characters
5. Normalizes whitespace

In [13]:
import re

# Define text cleaning function
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\S+", "", text)              # Remove URLs
    text = re.sub(r"<[^>]+>", "", text)                     # Remove HTML tags
    text = re.sub(r"[^\x20-\x7E]", " ", text)               # Remove non-ASCII/control characters
    text = re.sub(r"\s+", " ", text)                        # Normalize whitespace
    return text

# Apply cleaning
fnn_df["article_cleaned"] = fnn_df["article"].apply(clean_text)

# Preview before/after
fnn_df[["article", "article_cleaned"]].sample(3)

,article,article_cleaned
12309,VET APPROVED REVIEWED & FACT-CHECKED BY Dr. Ch...,vet approved reviewed & fact-checked by dr. ch...
17199,It’s not a Jersey Shore reunion without an awk...,it s not a jersey shore reunion without an awk...
13354,McDonald’s has pulled its new advert from TV s...,mcdonald s has pulled its new advert from tv s...


### d. Robust Date Cleaning

#### Why This Matters  
The `date` field in FakeNewsNet comes in various formats (ISO timestamps, partial dates, unexpected timezone info).  
Inconsistent or invalid dates can break pipelines that rely on date-based features or retrieval systems that match by publishing date.

---

#### Our plan  
1. Use `dateutil.parser` to safely parse each `date` string.  
1. Format valid dates to a **`DD/MM/YYYY`** structure.  
1. Replace any unparseable or missing dates with a clear placeholder (`"UNKNOWN_DATE"`).  
1. Preview the results to confirm the cleaned dates make sense.


In [14]:
from dateutil import parser

# Define a robust parser
def robust_parse_date(x):
    try:
        return parser.parse(str(x))
    except:
        return pd.NaT

# Apply parsing
fnn_df["date_parsed"] = fnn_df["date"].apply(robust_parse_date)

# Format to DD/MM/YYYY
fnn_df["date_cleaned"] = fnn_df["date_parsed"].apply(
    lambda x: x.strftime("%d/%m/%Y") if pd.notnull(x) else "UNKNOWN_DATE"
)

# Preview before/after
fnn_df[["date", "date_cleaned"]].sample(5)

,date,date_cleaned
11180,UNKNOWN_DATE,UNKNOWN_DATE
14325,UNKNOWN_DATE,UNKNOWN_DATE
20020,2017-08-31,31/08/2017
17008,UNKNOWN_DATE,UNKNOWN_DATE
5801,2017-12-04,04/12/2017


## Final Step: Save the Cleaned Dataset

Now that we've:
- Removed invalid or empty articles
- Handled missing fields (`url`, `date`)
- Filtered outliers
- Extracted `source_domain`
- Cleaned the article text
- Standardized the date format

…it’s time to save our cleaned dataset!

Keep only the **essential, well-structured columns**:
- `id`
- `title`
- `label`
- `source`
- `source_domain`
- `article_cleaned`
- `date_cleaned`


In [20]:
# Define columns to keep
final_columns = [
    "id",
    "title",
    "label",
    "dataset_source",
    "source_domain",
    "article_cleaned",
    "date_cleaned",
    *[column for column in fnn_df.columns if "source_" in column]
]


# Create final cleaned DataFrame
fnn_cleaned = fnn_df[final_columns].copy()

# Save to the specified location
output_path = base_path / Path("../../data/processed/fnn_cleaned.csv")
fnn_cleaned.to_csv(output_path, index=False)

print(f"Saved cleaned dataset with {len(fnn_cleaned):,} rows to '{output_path}'")

Saved cleaned dataset with 13,892 rows to '/content/drive/MyDrive/MASAID/AI_Project_Course/Misinformation/repo/week_1/sessions/../../data/processed/fnn_cleaned_new.csv'


## Package into Functions

> 📝: This function will be placed into `src`

It is now time to package these functionalities into functions and place them in our `src` folder.

We will build the `fakenewsnet_data_cleaning_pipeline` by calling three seperate functions: `remove`, `impute`, and `transform`, additionally with the functionality to save our data.

In [ ]:
import pandas as pd
import re
from pathlib import Path

def fakenewsnet_data_cleaning_pipeline(df, save_path=None):
    """
    Clean FakeNewsNet text data.

    Steps:
      1. Remove missing or empty articles
      2. Clean article text (URLs, HTML tags, non-ASCII chars, etc.)
      3. Optionally save cleaned data

    Args:
        df (pd.DataFrame): Raw FakeNewsNet dataframe with an 'article' column.
        save_path (str or Path, optional): Path to save cleaned CSV. If None, no file is saved.

    Returns:
        pd.DataFrame: Cleaned dataframe with a new 'article_cleaned' column.
    """

    def remove(df):
        # --- Step 1: Remove missing/empty ---
        df = df[
            df["article"].notnull() &
            (df["article"].astype(str).str.strip() != "")
        ].copy()

        # --- Step 1: Remove outliers
        df["article_length"] = df["article"].astype(str).apply(len)

        df = df[
            (df["article_length"] >= 250) &
            (df["article_length"] <= 30000)
        ].copy()

        # --- Step 3: Remove duplicate entries
        df = df.drop_duplicates(subset=["title", "article"], keep="first")

        return df

    def impute(df):

        # Fill URL and date with default placeholder
        df["url"] = df["url"].fillna("UNKNOWN_URL")

        df["date"] = df["date"].fillna("UNKNOWN_DATE")

        from urllib.parse import urlparse

        return df


    def transform(df):

        # --- Step 1: Extract domain ---
        def extract_domain(url):
            # Treat placeholder as unknown
            if url == "UNKNOWN_URL" or not isinstance(url, str) or url.strip() == "":
                return "unknown"
            try:
                cleaned = url.strip()
                if not cleaned.startswith(("http://", "https://")):
                    cleaned = "https://" + cleaned  # default scheme
                netloc = urlparse(cleaned).netloc
                return netloc.replace("www.", "") if netloc else "invalid"
            except:
                return "invalid"

        # --- Step 2: Apply One Hot Encoding on the source domains ---
        # Apply source domain extraction
        df["source_domain"] = df["url"].apply(extract_domain)

        # Count top 15 domains
        top_domains = df["source_domain"].value_counts().nlargest(15).index.tolist()

        # Assign 'Other' to all less common domains
        df["source_domain_grouped"] = df["source_domain"].apply(
            lambda x: x if x in top_domains else "Other"
        )

        # One-hot encode the grouped domain column
        source_dummies = pd.get_dummies(df["source_domain_grouped"], prefix="source")

        # Add to main DataFrame
        df = pd.concat([df, source_dummies], axis=1)

        # --- Step 3: Clean text ---
        def clean_text(text):
            text = str(text).lower().strip()
            text = re.sub(r"https?://\S+|www\.\S+", "", text)   # Remove URLs
            text = re.sub(r"<[^>]+>", "", text)                 # Remove HTML tags
            text = re.sub(r"[^\x20-\x7E]", " ", text)           # Remove non-ASCII/control chars
            text = re.sub(r"\s+", " ", text).strip()            # Normalize whitespace
            return text

        df["article_cleaned"] = df["article"].apply(clean_text)

        # Step 4: Parse dates
        from dateutil import parser
        import pandas as pd

        # Define a robust parser
        def robust_parse_date(x):
            try:
                return parser.parse(str(x))
            except:
                return pd.NaT

        # Apply parsing
        df["date_parsed"] = df["date"].apply(robust_parse_date)

        # Format to DD/MM/YYYY
        df["date_cleaned"] = df["date_parsed"].apply(
            lambda x: x.strftime("%d/%m/%Y") if pd.notnull(x) else "UNKNOWN_DATE"
        )

        return df

    if len(df["labels"].unique()) > 2:
        df['label'] = df['label'].replace({'0': 'real', '1': 'fake'})

    # Apply the 3 steps of the pipeline
    df = remove(df)
    df = impute(df)
    df = transform(df)

    # Save if requested
    if save_path is not None:
        save_path = Path(save_path)
        if hasattr(save_path, 'parent'):
          save_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(save_path, index=False)
        print(f"Cleaned data saved to {save_path}")

    print(f"Cleaned {len(df):,} rows total")

    return df


Just like in the previous session, we need to place it into the correct file. In this case, that would be `/src/misinformation_detection/data/data_cleaner.py` file.

And again, we need to add the function into the list of exposed functions in the `/src/misinformation_detection/data/__init__.py` by adding the line

``` python
from .data_cleaner import load_fakenewsnet_data
```  

## [💎 Additional Credit] Pytest

Again, all test code is present in the `tests` folder